# DEMO 13 Part 3: Advanced Datasets, Filters, and Interactivity

This demo continues from Parts 1 & 2. You should already have:
- A dashboard named **Sales Performance Dashboard**
- Multiple datasets (gold_sales, KPI Summary, Revenue by Region, Monthly Revenue Trend, etc.)
- Visualizations on the canvas (counters, bar chart, line chart, table)

Now we'll add **interactivity** — the features that make dashboards dynamic:
1. **Filter widgets** — let viewers slice data (like Power BI slicers)
2. **Parameters** — dynamic SQL placeholders (like Power BI parameters)
3. **Cross-filtering** — click a data point to filter other visuals
4. **Drill-through** — navigate to a detail page from a data point
5. **Custom calculations** — define measures and dimensions without changing SQL

| Power BI Feature | Databricks Equivalent |
| --- | --- |
| Slicer visual | **Filter widget** (interactive) |
| Visual-level filter (hidden) | **Widget-level filter** (static, author-set) |
| Report-level filter | **Global filter** |
| Page-level filter | **Page-level filter** |
| Parameters (Power Query / DAX) | **Query parameters** (`:param_name`) |
| Cross-highlight / cross-filter | **Cross-filtering** |
| Drill-through | **Drill-through** |
| DAX calculated column | **Calculated dimension** |
| DAX measure | **Calculated measure** |

> **Prerequisite:** Open your **Sales Performance Dashboard** in draft mode to continue.

---

## Understanding Filter Scopes

Before adding filters, understand how they differ in scope:

| Scope | Who Controls It | Applies To | Power BI Equivalent |
| --- | --- | --- | --- |
| **Global filter** | Viewer (interactive) | ALL pages, all widgets | Report-level filter / Synced slicer |
| **Page-level filter** | Viewer (interactive) | All widgets on ONE page | Page-level filter |
| **Widget-level filter** | Author (static) | ONE widget only | Visual-level filter (hidden from viewer) |

**Key distinction:**
- Global and page-level filters are **interactive** — viewers can change them
- Widget-level filters are **static** — set by the author, invisible to viewers

**Performance note:** For smaller datasets (≤ 100K rows / 100 MB), filters apply in the browser for instant responsiveness. For larger datasets, the filter is added to the SQL query and re-executed on the warehouse.

---

## Part 1: Add a Filter Widget (Interactive Slicer)

Filter widgets let viewers slice data — just like slicers in Power BI.

### Step 1: Add a filter widget
1. On the **Canvas** tab, click **Add a widget**
2. Select **Filter**

### Step 2: Configure the filter
1. In the configuration panel:
   - **Title:** "Region"
   - **Dataset:** Select `gold_sales`
   - **Column:** Select `region`
   - **Type:** Dropdown (single-select) or Multi-select
2. The filter widget appears on the canvas showing all unique values from that column

### Step 3: Connect to visualizations
- By default, the filter applies to **all visualizations on the same page** that use the same dataset (or a dataset with a matching column name)
- No manual "binding" needed — it works automatically based on column name matching

### Step 4: Add more filters
Repeat the steps to add filters for:
- **Product Category** (column: `product_category`)
- **Customer Segment** (column: `customer_segment`)
- **Date Range** (column: `order_date`) — this creates a date picker!

### Step 5: Arrange filter widgets
- Drag filters to the top of the page (common convention)
- Resize to fit side-by-side in a row

> **Power BI parallel:** This is identical to adding Slicer visuals. The auto-connection behavior is like Power BI's implicit relationships — if column names match, it just works.

---

## Part 2: Global Filters vs Page-Level Filters

### Make a filter global (applies to ALL pages)
1. Click on a filter widget you already placed
2. In the configuration panel, find the **Scope** setting
3. Change it from **Page** to **Global**
4. This filter now affects every visualization across all dashboard pages

### When to use each:

| Use Case | Scope |
| --- | --- |
| Date range that should apply everywhere | **Global** |
| Region filter that only applies to the summary page | **Page** |
| A filter specific to one chart (e.g., "Top 10 only") | **Widget-level** (static) |

### Add a widget-level filter (static)
Widget-level filters are set by the dashboard author and cannot be changed by viewers:

1. Click a visualization widget (e.g., the bar chart)
2. In the configuration panel, find **Filters** section
3. Click **+ Add filter**
4. Select a column (e.g., `channel`) and set a fixed value (e.g., "Online")
5. This visualization now only shows Online data — viewers cannot change it

> **Use case:** Show two identical charts side-by-side — one filtered to "Online", one to "Retail Store" — for comparison.

> **Power BI parallel:** Widget-level filters are like visual-level filters in the Power BI Filter pane (the ones viewers never see).

---

## Part 3: Query Parameters — Dynamic SQL for Dashboards

Parameters take filtering further: instead of filtering on existing columns, they inject values **directly into the SQL query**.

### Why parameters?

| Without Parameters | With Parameters |
| --- | --- |
| Write one query for East region | Write ONE query with `:region` placeholder |
| Copy/paste for West region | Viewer selects the region at runtime |
| Maintain multiple versions | Single query serves all cases |
| Risk of SQL injection | Strongly typed, injection-safe |

### How it works:
1. You define a parameter in your dataset SQL (e.g., `:selected_region`)
2. A filter widget is bound to that parameter
3. When the viewer selects a value, it's injected into the SQL query
4. The query re-runs with the substituted value

### When to use parameters vs column filters:

| Scenario | Use |
| --- | --- |
| Filter on a column already in the dataset | **Column filter** (simpler) |
| Filter on a value not in the result set | **Parameter** |
| Need conditional logic (IF/CASE based on selection) | **Parameter** |
| Dynamic table or column selection | **Parameter** |

---

## Step-by-Step: Add a Parameter to a Dataset

### Step 1: Modify a dataset's SQL
1. Click the **Data** tab
2. Select (or create) a dataset — for example, a new one called **Filtered Revenue by Category**
3. Write the SQL using the `:parameter_name` syntax:

```sql
-- Revenue by category, filtered by a dynamic region
SELECT
  product_category,
  ROUND(SUM(net_revenue), 2) AS total_revenue,
  COUNT(order_id) AS order_count
FROM main.demo_dashboards_<your_username>.gold_sales
WHERE region = :selected_region
GROUP BY product_category
ORDER BY total_revenue DESC
```

4. As soon as you type `:selected_region`, Databricks detects it and shows a **parameter input** above the query editor
5. Type a test value (e.g., `Northeast`) and click **Run** to verify

### Step 2: Connect a filter widget to the parameter
1. Switch to the **Canvas** tab
2. Add a new **Filter** widget
3. In the configuration panel:
   - **Type:** Select **Query parameter**
   - **Parameter:** Choose `selected_region`
   - **Values from:** Select the `gold_sales` dataset, column `region` (this populates the dropdown)
4. The filter widget now shows all unique region values as options

### Step 3: Test it
1. Select a region from the dropdown
2. The visualization bound to the **Filtered Revenue by Category** dataset updates automatically
3. The SQL re-executes with the selected value substituted into the WHERE clause

> **Power BI parallel:** This is like a Power BI parameter bound to a slicer — but more powerful because you can use it anywhere in SQL (WHERE, HAVING, CASE expressions, even table names).

---

## Part 4: Cross-Filtering

Cross-filtering lets viewers click a data point in one chart to filter all other charts on the same page.

### How to enable:
1. Cross-filtering is **enabled by default** on most visualization types
2. No configuration needed — it works automatically for widgets that share the same dataset or have matching column names

### How it works:
1. Viewer clicks a bar in the "Revenue by Region" chart (e.g., clicks "Northeast")
2. All other visualizations on the same page automatically filter to show only Northeast data
3. Click again to deselect (or click a different bar)

### Which widgets support cross-filtering:
- Bar charts ✔️
- Line charts ✔️
- Pie charts ✔️
- Area charts ✔️
- Scatter plots ✔️
- Tables (click a row) ✔️
- Counters ✘ (no clickable data points)

### Disable cross-filtering (if needed):
1. Click the visualization widget
2. In the configuration panel, find **Cross-filtering**
3. Toggle it off for that specific widget

> **Power BI parallel:** Identical to Power BI's cross-filtering behavior. When you click a bar in one visual, other visuals filter. The main difference: Power BI also supports "cross-highlighting" (dimming non-matching data). Databricks currently supports filtering only.

---

## Part 5: Drill-Through

Drill-through lets viewers click a data point and navigate to a **detail page** filtered by their selection.

### Step 1: Create a target page
1. Click **+ Add page** at the bottom of the canvas
2. Rename it to **Region Detail** (right-click the tab)
3. Add visualizations to this page that show region-level detail (e.g., monthly trend for a specific region, top products in that region)

### Step 2: Configure drill-through on the target page
1. On the **Region Detail** page, look for the **Drill-through** configuration
2. Set the **Drill-through field** to `region`
3. This tells the page: "When someone drills through to me, filter everything by the region value they clicked"

### Step 3: Use drill-through from the source page
1. Navigate back to your main page
2. On any visualization that contains `region` data:
   - Viewer right-clicks (or clicks) a specific region data point
   - A **drill-through** option appears
   - Clicking it navigates to the Region Detail page, filtered to that region

### Step 4: Add a Back button
- The target page automatically gets a **Back** button
- Viewers click it to return to the source page

> **Power BI parallel:** Same concept as Power BI drill-through. You configure a target page with a drill-through field, and source visuals automatically offer the drill-through action on matching data points.

---

## Part 6: Custom Calculations

Custom calculations let you define dynamic metrics and transformations **without modifying the dataset's SQL query**. They operate on the dataset results.

### Two types:

| Type | What It Is | Power BI Equivalent | Example |
| --- | --- | --- | --- |
| **Calculated Measure** | Aggregated value | DAX Measure | `SUM(net_revenue) - SUM(cost)` = Profit |
| **Calculated Dimension** | Unaggregated transformation | Calculated Column | `CASE WHEN quantity > 3 THEN 'Bulk' ELSE 'Single' END` |

### Key features:
- Up to **200 custom calculations** per dataset
- Calculated measures are **dynamic** — they update when filters change
- Supports **AGGREGATE OVER** for range-based calculations (e.g., trailing 7-day average)
- No need to edit the original SQL query

---

## Step-by-Step: Add Custom Calculations

### Add a Calculated Measure
1. In the **Data** tab, click on a dataset (e.g., `gold_sales`)
2. Look for the **Custom calculations** section (or a "+" near the column list)
3. Click **+ Add calculated measure**
4. Enter:
   - **Name:** `Profit`
   - **Expression:** `SUM(net_revenue) - SUM(cost)`
5. Click **Save**
6. The new `Profit` measure now appears in the column list and can be used in visualizations

### Add a Calculated Dimension
1. Same dataset, click **+ Add calculated dimension**
2. Enter:
   - **Name:** `Order Size`
   - **Expression:** `CASE WHEN quantity > 3 THEN 'Bulk' ELSE 'Single' END`
3. Click **Save**
4. You can now use `Order Size` as a grouping field in charts

### Use AGGREGATE OVER for rolling calculations
For time-based running totals or trailing windows:
1. Add a calculated measure:
   - **Name:** `7-Day Avg Revenue`
   - **Expression:** `AGGREGATE OVER (AVG(net_revenue), order_date, -7, 0)`

This computes a trailing 7-day average profit margin for each data point.

> **Power BI parallel:** Calculated measures = DAX measures. Calculated dimensions = DAX calculated columns. AGGREGATE OVER = DAX time-intelligence functions (DATESINPERIOD, etc.).

---

---

## Summary

| Feature | What It Does | When to Use |
| --- | --- | --- |
| **Filter widget** | Interactive slicer for viewers | Let users slice by category, date, region |
| **Global filter** | Applies across all pages | Date range, user segment — universal context |
| **Widget-level filter** | Static, author-set | Side-by-side comparisons (same chart, different filter) |
| **Parameter** | Injects value into SQL `:param` | Dynamic SQL, conditional logic, non-column filters |
| **Cross-filtering** | Click to filter other visuals | Exploratory dashboards — let users dig in |
| **Drill-through** | Navigate to detail page | Summary → Detail workflows |
| **Calculated measure** | Dynamic aggregation | Ad-hoc metrics without editing SQL |
| **Calculated dimension** | Row-level transformation | Categorization, bucketing, formatting |

### Publishing Reminder

When your dashboard is ready:
1. Click **Publish** (top right)
2. Choose credential mode:
   - **Share data permission** (default) — viewers use publisher's data access
   - **Individual data permission** — viewers use their own credentials
3. The published version is a **frozen snapshot** — updates require re-publishing

> A **companion Genie Space** is automatically created when you publish. Viewers can click "Ask Genie" to ask natural language questions about the dashboard data.

**Next:** We'll configure a Genie Space and explore its knowledge store.